# Data Augmentation Notebook

This notebook performs data augmentation on the training data from the `raw` folder and saves the augmented data to the `augmented` folder.

## Augmentation Techniques:
- Rotation
- Zooming (Scaling)
- Sheering
- Translation
- Deformation grid (elastic deformation)


In [1]:
# Setup and Imports
import sys
from pathlib import Path
import pickle
import gzip
import numpy as np
from tqdm.notebook import tqdm
import random
from scipy.ndimage import rotate
import cv2

import elasticdeform

# Path setup
current_dir = Path.cwd()
BASE_PATH = current_dir.parent

RAW_DATA_PATH = BASE_PATH / 'data' / 'raw'
AUGMENTED_DATA_PATH = BASE_PATH / 'data' / 'augmented'

# Create augmented directory if it doesn't exist
AUGMENTED_DATA_PATH.mkdir(parents=True, exist_ok=True)

print(f"Base path: {BASE_PATH}")
print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Augmented data path: {AUGMENTED_DATA_PATH}")


Base path: /users/lmantel/lab
Raw data path: /users/lmantel/lab/data/raw
Augmented data path: /users/lmantel/lab/data/augmented


## Augmentation Functions

Each function applies a specific transformation to both the image and mask to keep them aligned.


In [2]:
def apply_rotation(image, mask, angle_range=(-15, 15)):
    """
    Apply rotation augmentation to image and mask.
    
    Args:
        image: numpy array of shape (H, W)
        mask: numpy array of shape (H, W), boolean
        angle_range: tuple of (min_angle, max_angle) in degrees
    
    Returns:
        rotated_image, rotated_mask
    """
    angle = random.uniform(angle_range[0], angle_range[1])
    
    # Rotate image
    rotated_img = rotate(image, angle, axes=(0, 1), reshape=False, order=1, mode='constant', cval=0)
    rotated_img = np.clip(rotated_img, 0, 255).astype(np.uint8)
    
    # Rotate mask with nearest neighbor interpolation to preserve binary values
    rotated_mask = rotate(mask.astype(float), angle, axes=(0, 1), reshape=False, order=0, mode='constant', cval=0)
    rotated_mask = (rotated_mask > 0.5).astype(bool)
    
    return rotated_img, rotated_mask


In [3]:
def apply_zooming(image, mask, zoom_range=(0.9, 1.1)):
    """
    Apply zooming (scaling) augmentation to image and mask.
    
    Args:
        image: numpy array of shape (H, W)
        mask: numpy array of shape (H, W), boolean
        zoom_range: tuple of (min_zoom, max_zoom) factors
    
    Returns:
        zoomed_image, zoomed_mask
    """
    zoom_factor = random.uniform(zoom_range[0], zoom_range[1])
    H, W = image.shape
    
    # Calculate new dimensions
    new_H, new_W = int(H * zoom_factor), int(W * zoom_factor)
    
    # Resize image
    zoomed_img = cv2.resize(image, (new_W, new_H), interpolation=cv2.INTER_LINEAR)
    zoomed_mask = cv2.resize(mask.astype(np.uint8), (new_W, new_H), interpolation=cv2.INTER_NEAREST)
    
    # Crop or pad to original size
    if zoom_factor > 1.0:
        # Crop from center
        start_h = (new_H - H) // 2
        start_w = (new_W - W) // 2
        zoomed_img = zoomed_img[start_h:start_h+H, start_w:start_w+W]
        zoomed_mask = zoomed_mask[start_h:start_h+H, start_w:start_w+W]
    else:
        # Pad with zeros
        pad_h = (H - new_H) // 2
        pad_w = (W - new_W) // 2
        zoomed_img = np.pad(zoomed_img, ((pad_h, H-new_H-pad_h), (pad_w, W-new_W-pad_w)), mode='constant', constant_values=0)
        zoomed_mask = np.pad(zoomed_mask, ((pad_h, H-new_H-pad_h), (pad_w, W-new_W-pad_w)), mode='constant', constant_values=0)
    
    zoomed_mask = (zoomed_mask > 0.5).astype(bool)
    return zoomed_img, zoomed_mask


In [4]:
def apply_sheering(image, mask, shear_range=(-0.2, 0.2)):
    """
    Apply sheering augmentation to image and mask.
    
    Args:
        image: numpy array of shape (H, W)
        mask: numpy array of shape (H, W), boolean
        shear_range: tuple of (min_shear, max_shear) factors
    
    Returns:
        sheered_image, sheered_mask
    """
    shear = random.uniform(shear_range[0], shear_range[1])
    H, W = image.shape
    
    # Create affine transformation matrix for sheering
    # Horizontal sheering
    if random.random() > 0.5:
        # Shear along x-axis
        transform_matrix = np.array([[1, shear, 0],
                                     [0, 1, 0],
                                     [0, 0, 1]], dtype=np.float32)
    else:
        # Shear along y-axis
        transform_matrix = np.array([[1, 0, 0],
                                     [shear, 1, 0],
                                     [0, 0, 1]], dtype=np.float32)
    
    # Apply transformation
    sheered_img = cv2.warpAffine(image, transform_matrix[:2], (W, H), 
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    sheered_mask = cv2.warpAffine(mask.astype(np.uint8), transform_matrix[:2], (W, H),
                                  flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    sheered_mask = (sheered_mask > 0.5).astype(bool)
    
    return sheered_img, sheered_mask


In [5]:
def apply_translation(image, mask, translate_range=(-10, 10)):
    """
    Apply translation augmentation to image and mask.
    
    Args:
        image: numpy array of shape (H, W)
        mask: numpy array of shape (H, W), boolean
        translate_range: tuple of (min_translate, max_translate) in pixels
    
    Returns:
        translated_image, translated_mask
    """
    H, W = image.shape
    tx = random.randint(translate_range[0], translate_range[1])
    ty = random.randint(translate_range[0], translate_range[1])
    
    # Create translation matrix
    transform_matrix = np.array([[1, 0, tx],
                                 [0, 1, ty]], dtype=np.float32)
    
    # Apply transformation
    translated_img = cv2.warpAffine(image, transform_matrix, (W, H),
                                    flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    translated_mask = cv2.warpAffine(mask.astype(np.uint8), transform_matrix, (W, H),
                                     flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    translated_mask = (translated_mask > 0.5).astype(bool)
    
    return translated_img, translated_mask


In [6]:
def apply_deformation_grid(image, mask, sigma=5, points=3, alpha=50):
    """
    Apply elastic deformation using deformation grid.
    
    Args:
        image: numpy array of shape (H, W)
        mask: numpy array of shape (H, W), boolean
        sigma: standard deviation of Gaussian filter for smooth deformation
        points: number of control points for deformation grid
        alpha: scaling factor for deformation strength
    
    Returns:
        deformed_image, deformed_mask
    """
    # Stack image and mask for simultaneous deformation
    # elasticdeform expects shape (H, W, C) or (H, W)
    # We'll apply to each separately to handle different dtypes
    
    # Apply deformation to image
    deformed_img = elasticdeform.deform_random_grid(image, sigma=sigma, points=points, 
                                                     mode='constant', cval=0, order=1)
    deformed_img = np.clip(deformed_img, 0, 255).astype(np.uint8)
    
    # Apply same deformation to mask (use order=0 for nearest neighbor)
    deformed_mask = elasticdeform.deform_random_grid(mask.astype(float), sigma=sigma, points=points,
                                                      mode='constant', cval=0, order=0)
    deformed_mask = (deformed_mask > 0.5).astype(bool)
    
    return deformed_img, deformed_mask


## Main Augmentation Function

This function applies random augmentations to a single sample (video + labels).


In [7]:
def augment_sample(sample, augmentation_prob=0.5):
    """
    Apply random augmentations to a sample.
    
    Args:
        sample: dict with keys 'video', 'label', 'box', 'name', 'frames', 'dataset'
        augmentation_prob: probability of applying each augmentation
    
    Returns:
        augmented_sample: dict with same structure as input
    """
    augmented_sample = sample.copy()
    video = sample['video'].copy()  # shape: (H, W, T)
    label = sample['label'].copy()   # shape: (H, W, T)
    box = sample['box'].copy()       # shape: (H, W)
    
    H, W, T = video.shape
    
    # Apply augmentations frame by frame to keep consistency
    # We apply the same transformation to all frames in a sample
    # Choose which augmentations to apply
    apply_rot = random.random() < augmentation_prob
    apply_zoom = random.random() < augmentation_prob
    apply_shear = random.random() < augmentation_prob
    apply_trans = random.random() < augmentation_prob
    apply_def = random.random() < augmentation_prob
    
    # Process each frame
    for t in range(T):
        frame = video[:, :, t]
        mask = label[:, :, t]
        
        # Apply augmentations in sequence
        if apply_rot:
            frame, mask = apply_rotation(frame, mask)
        
        if apply_zoom:
            frame, mask = apply_zooming(frame, mask)
        
        if apply_shear:
            frame, mask = apply_sheering(frame, mask)
        
        if apply_trans:
            frame, mask = apply_translation(frame, mask)
        
        if apply_def:
            frame, mask = apply_deformation_grid(frame, mask)
        
        video[:, :, t] = frame
        label[:, :, t] = mask
    
    # Apply same transformations to box (2D mask)
    box_frame = box.copy()
    box_mask = box.copy()
    
    if apply_rot:
        box_frame, box_mask = apply_rotation(box_frame.astype(np.uint8) * 255, box_mask)
        box = box_mask
    
    if apply_zoom:
        box_frame, box_mask = apply_zooming(box_frame.astype(np.uint8), box_mask)
        box = box_mask
    
    if apply_shear:
        box_frame, box_mask = apply_sheering(box_frame.astype(np.uint8), box_mask)
        box = box_mask
    
    if apply_trans:
        box_frame, box_mask = apply_translation(box_frame.astype(np.uint8), box_mask)
        box = box_mask
    
    if apply_def:
        box_frame, box_mask = apply_deformation_grid(box_frame.astype(np.uint8), box_mask)
        box = box_mask
    
    augmented_sample['video'] = video
    augmented_sample['label'] = label
    augmented_sample['box'] = box
    
    return augmented_sample


## Helper Functions

Functions for loading and saving gzipped pickle files.


In [8]:
def load_zipped_pickle(filename):
    """Load a gzipped pickle file."""
    with gzip.open(filename, 'rb') as f:
        loaded_object = pickle.load(f)
        return loaded_object

def save_zipped_pickle(obj, filename):
    """Save an object to a gzipped pickle file."""
    with gzip.open(filename, 'wb') as f:
        pickle.dump(obj, f, 2)


In [9]:
# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Load training data
print("Loading training data...")
train_data = load_zipped_pickle(str(RAW_DATA_PATH / 'train.pkl'))
print(f"Loaded {len(train_data)} training samples")

# Load test data (optional, for reference)
print("Loading test data...")
test_data = load_zipped_pickle(str(RAW_DATA_PATH / 'test.pkl'))
print(f"Loaded {len(test_data)} test samples")


Loading training data...
Loaded 65 training samples
Loading test data...
Loaded 20 test samples


In [10]:
# Configuration
NUM_AUGMENTATIONS = 2  # Number of augmented versions per original sample
AUGMENTATION_PROB = 0.7  # Probability of applying each augmentation technique

print(f"Configuration:")
print(f"  - Number of augmentations per sample: {NUM_AUGMENTATIONS}")
print(f"  - Augmentation probability: {AUGMENTATION_PROB}")
print(f"  - Total samples to create: {len(train_data) * (1 + NUM_AUGMENTATIONS)}")


Configuration:
  - Number of augmentations per sample: 2
  - Augmentation probability: 0.7
  - Total samples to create: 195


In [11]:
# Create augmented dataset
augmented_train_data = []

# Start with original data
print("Adding original training samples...")
for sample in tqdm(train_data, desc="Original samples"):
    augmented_train_data.append(sample.copy())

# Add augmented samples
print(f"Creating {NUM_AUGMENTATIONS} augmented versions per sample...")
for sample in tqdm(train_data, desc="Augmenting samples"):
    for aug_idx in range(NUM_AUGMENTATIONS):
        augmented_sample = augment_sample(sample, augmentation_prob=AUGMENTATION_PROB)
        # Update name to indicate augmentation
        augmented_sample['name'] = f"{sample['name']}_aug{aug_idx+1}"
        augmented_train_data.append(augmented_sample)

print(f"\nTotal augmented training samples: {len(augmented_train_data)}")
print(f"  - Original: {len(train_data)}")
print(f"  - Augmented: {len(augmented_train_data) - len(train_data)}")


Adding original training samples...


Original samples:   0%|          | 0/65 [00:00<?, ?it/s]

Creating 2 augmented versions per sample...


Augmenting samples:   0%|          | 0/65 [00:00<?, ?it/s]


Total augmented training samples: 195
  - Original: 65
  - Augmented: 130


In [ ]:
# Save augmented training data
output_path = AUGMENTED_DATA_PATH / 'train_augmented.pkl'
print(f"Saving augmented training data to {output_path}...")
save_zipped_pickle(augmented_train_data, str(output_path))
print("Done!")

# Also save test data (unchanged) for convenience
test_output_path = AUGMENTED_DATA_PATH / 'test.pkl'
print(f"Saving test data to {test_output_path}...")
save_zipped_pickle(test_data, str(test_output_path))
print("Done!")


Saving augmented training data to /users/lmantel/lab/data/augmented/train_augmented.pkl...


In [ ]:
# Verify saved data
print("Verifying saved augmented data...")
loaded_augmented = load_zipped_pickle(str(AUGMENTED_DATA_PATH / 'train_augmented.pkl'))
print(f"Loaded {len(loaded_augmented)} samples")

# Inspect a few samples
print("\nSample inspection:")
for i, sample in enumerate(loaded_augmented[:3]):
    print(f"\nSample {i}:")
    print(f"  Name: {sample.get('name')}")
    vid = sample.get('video')
    lab = sample.get('label')
    box = sample.get('box')
    print(f"  Video: shape={vid.shape}, dtype={vid.dtype}")
    print(f"  Label: shape={lab.shape}, dtype={lab.dtype}")
    print(f"  Box: shape={box.shape}, dtype={box.dtype}")
    print(f"  Frames: {sample.get('frames')}")
    print(f"  Dataset: {sample.get('dataset')}")


In [ ]:
# Visualize original vs augmented sample
import matplotlib.pyplot as plt

# Find an original sample and its augmented version
original_sample = train_data[0]
augmented_sample = None
for sample in loaded_augmented:
    if sample['name'].startswith(original_sample['name'] + '_aug'):
        augmented_sample = sample
        break

if augmented_sample is not None:
    # Get a frame from the middle
    frame_idx = original_sample['frames'][len(original_sample['frames'])//2]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Original
    axes[0, 0].imshow(original_sample['video'][:, :, frame_idx], cmap='gray')
    axes[0, 0].set_title(f"Original Video - Frame {frame_idx}")
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(original_sample['label'][:, :, frame_idx], cmap='gray')
    axes[0, 1].set_title("Original Label")
    axes[0, 1].axis('off')
    
    # Overlay
    overlay_orig = original_sample['video'][:, :, frame_idx].copy()
    overlay_orig = np.stack([overlay_orig, overlay_orig, overlay_orig], axis=-1)
    mask_orig = original_sample['label'][:, :, frame_idx]
    overlay_orig[:, :, 1] = np.where(mask_orig, overlay_orig[:, :, 1] * 0.5 + 255 * 0.5, overlay_orig[:, :, 1])
    axes[0, 2].imshow(overlay_orig.astype(np.uint8))
    axes[0, 2].set_title("Original Overlay")
    axes[0, 2].axis('off')
    
    # Augmented
    aug_frame_idx = augmented_sample['frames'][len(augmented_sample['frames'])//2] if len(augmented_sample['frames']) > 0 else frame_idx
    axes[1, 0].imshow(augmented_sample['video'][:, :, aug_frame_idx], cmap='gray')
    axes[1, 0].set_title(f"Augmented Video - Frame {aug_frame_idx}")
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(augmented_sample['label'][:, :, aug_frame_idx], cmap='gray')
    axes[1, 1].set_title("Augmented Label")
    axes[1, 1].axis('off')
    
    # Overlay
    overlay_aug = augmented_sample['video'][:, :, aug_frame_idx].copy()
    overlay_aug = np.stack([overlay_aug, overlay_aug, overlay_aug], axis=-1)
    mask_aug = augmented_sample['label'][:, :, aug_frame_idx]
    overlay_aug[:, :, 1] = np.where(mask_aug, overlay_aug[:, :, 1] * 0.5 + 255 * 0.5, overlay_aug[:, :, 1])
    axes[1, 2].imshow(overlay_aug.astype(np.uint8))
    axes[1, 2].set_title("Augmented Overlay")
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("Could not find matching augmented sample for visualization")


In [ ]:
# Helper functions for loading and saving
def load_zipped_pickle(filename):
    """Load a gzipped pickle file."""
    with gzip.open(filename, 'rb') as f:
        loaded_object = pickle.load(f)
        return loaded_object

def save_zipped_pickle(obj, filename):
    """Save an object to a gzipped pickle file."""
    with gzip.open(filename, 'wb') as f:
        pickle.dump(obj, f, 2)
